# PART IV QUALITY - Compare Human and AI Reviews (Fixed + Expanded)

This notebook fixes the earlier issues and adds expanded analyses for:
- quality comparison between human and AI proposals
- AI-reviewer bias diagnostics
- proxy validity of AI reviews vs human experts

Major fixes implemented:
1. Single, top-to-bottom reproducible execution order
2. Robust proposal matching (exact + fuzzy fallback) with diagnostics
3. No duplicated `pair_df` construction
4. Better `review_text` joining with separators (`"\n\n"`)
5. No duplicate `human-all` rows contaminating evaluator-level analyses
6. Proposal-level inference (to reduce non-independence from pairwise combinations)
7. Multiple-testing correction (Benjamini-Hochberg FDR)
8. Sensitivity analysis for exact-only vs exact+fuzzy matching


## 0) Environment setup (run once if needed)


In [27]:
# Uncomment if needed
# %pip install -q numpy pandas scipy matplotlib seaborn openpyxl textblob transformers torch scikit-learn statsmodels krippendorff


## 1) Imports, paths, and constants


In [28]:
import json
import re
import itertools
from pathlib import Path
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu, kruskal, wilcoxon, spearmanr, kendalltau, linregress
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob

import torch
from transformers import AutoTokenizer, AutoModel

sns.set_theme(style='whitegrid', context='talk')

AI_REVIEWS_PATH = Path('data/reviews/ai_reviews/ai_reviews_ncems_criteria_20260223_153411.json')
HUMAN_Y1_REVIEWS_PATH = Path('data/reviews/human_reviews/human_reviews_human-y1.xlsx')
OUTPUT_DIR = Path('results/figures/quality')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

colors = {
    'Human': '#DC143C',
    'claude-opus-4-5': '#4A90E2',
    'gemini-3-pro-preview': '#7B68EE',
    'gpt-5.2': '#FF8C00',
}

CRITERIA_ORDER = [
    'Relevance_to_Emergent_Phenomena',
    'Novelty_and_Significance',
    'Rigor_of_Approach',
    'Scope_and_Timeline',
    'Synthesis_Focus',
    'Data_Identification',
    'Open_Science_Commitment',
]

CRITERION_MAP = {
    'Relevance to Emergent Phenomena': 'Relevance_to_Emergent_Phenomena',
    'Novelty & Significance': 'Novelty_and_Significance',
    'Rigor of Approach': 'Rigor_of_Approach',
    'Scope & Timeline': 'Scope_and_Timeline',
    'Synthesis Focus': 'Synthesis_Focus',
    'Data Identification': 'Data_Identification',
    'Open Science Commitment': 'Open_Science_Commitment',
}

HUMAN_COL_MAP = {
    'scientific_merit_and_innovation_score': ['Relevance_to_Emergent_Phenomena', 'Novelty_and_Significance', 'Rigor_of_Approach'],
    'feasibility_score': ['Scope_and_Timeline'],
    'data_sources_and_limitations_score': ['Synthesis_Focus', 'Data_Identification'],
    'open_science_compliance_score': ['Open_Science_Commitment'],
}

SHARED_METRICS_4CAT = [
    'Scientific_Merit_and_Innovation',
    'Feasibility',
    'Data_Sources_and_Limitations',
    'Open_Science_Compliance',
]


## 2) Utility functions


In [29]:
def normalize_title(x: str) -> str:
    if pd.isna(x):
        return ''
    x = str(x).lower().strip()
    x = re.sub(r'[^a-z0-9]+', ' ', x)
    x = re.sub(r'\s+', ' ', x).strip()
    return x


def token_set_jaccard(a: str, b: str) -> float:
    sa = set(a.split())
    sb = set(b.split())
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def hybrid_similarity(a: str, b: str) -> float:
    seq = SequenceMatcher(None, a, b).ratio()
    jac = token_set_jaccard(a, b)
    return 0.7 * seq + 0.3 * jac


def cliffs_delta(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    gt = 0
    lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)
    return (gt - lt) / (len(x) * len(y))


def interpret_cliffs_delta(delta):
    if np.isnan(delta):
        return 'NA'
    ad = abs(delta)
    if ad < 0.147:
        return 'negligible'
    if ad < 0.33:
        return 'small'
    if ad < 0.474:
        return 'medium'
    return 'large'


def benjamini_hochberg(pvalues):
    p = np.array(pvalues, dtype=float)
    n = len(p)
    if n == 0:
        return p
    order = np.argsort(p)
    ranked = p[order]
    q = np.empty(n)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * n / rank
        prev = min(prev, val)
        q[i] = prev
    out = np.empty(n)
    out[order] = np.clip(q, 0, 1)
    return out


def add_bh_fdr(df, p_col='p_value', group_cols=None, out_col='q_value'):
    df = df.copy()
    if group_cols is None:
        df[out_col] = benjamini_hochberg(df[p_col].values)
        return df
    df[out_col] = np.nan
    for _, idx in df.groupby(group_cols).groups.items():
        idx = list(idx)
        df.loc[idx, out_col] = benjamini_hochberg(df.loc[idx, p_col].values)
    return df


def mannwhitney_summary(a, b, metric_name='metric'):
    a = pd.Series(a).dropna().to_numpy()
    b = pd.Series(b).dropna().to_numpy()
    if len(a) == 0 or len(b) == 0:
        return {
            'metric': metric_name,
            'n_group1': len(a),
            'n_group2': len(b),
            'u_stat': np.nan,
            'p_value': np.nan,
            'cliffs_delta': np.nan,
            'delta_magnitude': 'NA',
        }
    u, p = mannwhitneyu(a, b, alternative='two-sided')
    d = cliffs_delta(a, b)
    return {
        'metric': metric_name,
        'n_group1': len(a),
        'n_group2': len(b),
        'u_stat': u,
        'p_value': p,
        'cliffs_delta': d,
        'delta_magnitude': interpret_cliffs_delta(d),
    }


def sentiment_label(p):
    if p > 0.1:
        return 'positive'
    if p < -0.1:
        return 'negative'
    return 'neutral'


def sentiment_alignment(p1, p2):
    return 1 - (abs(p1 - p2) / 2)


def categorical_agreement(l1, l2):
    if l1 == l2:
        return 'agree'
    if {'positive', 'negative'} == {l1, l2}:
        return 'disagree'
    return 'partial'


def bootstrap_mean_diff_ci(x, y, n_boot=2000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    x = np.asarray(pd.Series(x).dropna().values, dtype=float)
    y = np.asarray(pd.Series(y).dropna().values, dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan, np.nan
    diffs = []
    for _ in range(n_boot):
        xb = rng.choice(x, size=len(x), replace=True)
        yb = rng.choice(y, size=len(y), replace=True)
        diffs.append(np.mean(xb) - np.mean(yb))
    diffs = np.array(diffs)
    alpha = (1 - ci) / 2
    lo = np.quantile(diffs, alpha)
    hi = np.quantile(diffs, 1 - alpha)
    return np.mean(x) - np.mean(y), lo, hi


def permutation_p_value_mean_diff(x, y, n_perm=5000, seed=42):
    rng = np.random.default_rng(seed)
    x = np.asarray(pd.Series(x).dropna().values, dtype=float)
    y = np.asarray(pd.Series(y).dropna().values, dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    obs = abs(np.mean(x) - np.mean(y))
    pooled = np.concatenate([x, y]).copy()
    n_x = len(x)
    count = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        xp = pooled[:n_x]
        yp = pooled[n_x:]
        if abs(np.mean(xp) - np.mean(yp)) >= obs:
            count += 1
    return (count + 1) / (n_perm + 1)


def icc2_1_2k(ratings_matrix):
    # ICC(2,1) and ICC(2,k) for complete items x raters matrix
    X = np.asarray(ratings_matrix, dtype=float)
    if np.isnan(X).any():
        return np.nan, np.nan
    n, k = X.shape
    if n < 2 or k < 2:
        return np.nan, np.nan

    mean_row = X.mean(axis=1, keepdims=True)
    mean_col = X.mean(axis=0, keepdims=True)
    grand = X.mean()

    ss_row = k * np.sum((mean_row - grand) ** 2)
    ss_col = n * np.sum((mean_col - grand) ** 2)
    ss_err = np.sum((X - mean_row - mean_col + grand) ** 2)

    ms_row = ss_row / (n - 1)
    ms_col = ss_col / (k - 1)
    ms_err = ss_err / ((n - 1) * (k - 1))

    icc21 = (ms_row - ms_err) / (ms_row + (k - 1) * ms_err + (k * (ms_col - ms_err) / n))
    icc2k = (ms_row - ms_err) / (ms_row + ((ms_col - ms_err) / n))
    return icc21, icc2k


def build_one_to_one_title_mapping(human_proposals, ai_proposals, fuzzy_threshold=0.70):
    h = human_proposals.copy().reset_index(drop=True)
    a = ai_proposals.copy().reset_index(drop=True)

    assigned_h = set()
    assigned_a = set()
    rows = []

    ai_by_norm = a.groupby('ai_title_norm')['ai_proposal_id'].apply(list).to_dict()
    for _, hr in h.iterrows():
        hp = hr['human_proposal_id']
        hn = hr['human_title_norm']
        candidates = ai_by_norm.get(hn, [])
        candidates = [x for x in candidates if x not in assigned_a]
        if len(candidates) == 1:
            ap = candidates[0]
            ar = a.loc[a['ai_proposal_id'] == ap].iloc[0]
            rows.append({
                'human_proposal_id': hp,
                'ai_proposal_id': ap,
                'human_title': hr['human_title'],
                'ai_title': ar['ai_title'],
                'human_title_norm': hn,
                'ai_title_norm': ar['ai_title_norm'],
                'similarity': 1.0,
                'match_method': 'exact',
            })
            assigned_h.add(hp)
            assigned_a.add(ap)

    h_un = h[~h['human_proposal_id'].isin(assigned_h)]
    a_un = a[~a['ai_proposal_id'].isin(assigned_a)]

    candidates = []
    for _, hr in h_un.iterrows():
        for _, ar in a_un.iterrows():
            sim = hybrid_similarity(hr['human_title_norm'], ar['ai_title_norm'])
            candidates.append((sim, hr['human_proposal_id'], ar['ai_proposal_id']))

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)
    used_h = set()
    used_a = set()

    for sim, hp, ap in candidates:
        if sim < fuzzy_threshold:
            break
        if hp in used_h or ap in used_a:
            continue
        hr = h.loc[h['human_proposal_id'] == hp].iloc[0]
        ar = a.loc[a['ai_proposal_id'] == ap].iloc[0]
        rows.append({
            'human_proposal_id': hp,
            'ai_proposal_id': ap,
            'human_title': hr['human_title'],
            'ai_title': ar['ai_title'],
            'human_title_norm': hr['human_title_norm'],
            'ai_title_norm': ar['ai_title_norm'],
            'similarity': sim,
            'match_method': 'fuzzy',
        })
        used_h.add(hp)
        used_a.add(ap)

    mapping = pd.DataFrame(rows)

    h_unmatched = sorted(set(h['human_proposal_id']) - set(mapping['human_proposal_id']))
    a_unmatched = sorted(set(a['ai_proposal_id']) - set(mapping['ai_proposal_id']))

    diagnostics = {
        'n_human_proposals': int(h.shape[0]),
        'n_ai_proposals': int(a.shape[0]),
        'n_matched': int(mapping.shape[0]),
        'n_exact': int((mapping['match_method'] == 'exact').sum()) if not mapping.empty else 0,
        'n_fuzzy': int((mapping['match_method'] == 'fuzzy').sum()) if not mapping.empty else 0,
        'n_unmatched_human': int(len(h_unmatched)),
        'n_unmatched_ai': int(len(a_unmatched)),
    }

    return mapping, diagnostics, h_unmatched, a_unmatched


## 3) Load and flatten AI reviews


In [30]:
with open(AI_REVIEWS_PATH, 'r') as f:
    ai_payload = json.load(f)

rows = []
for r in ai_payload.get('reviews', []):
    ev = (r.get('evaluations') or {}).get('evaluation')
    if not isinstance(ev, dict):
        continue

    criteria = {k: np.nan for k in CRITERIA_ORDER}
    justifications = []

    for category in ev.get('criteria_scores', []) or []:
        for sc in category.get('subcriteria', []) or []:
            c_name = CRITERION_MAP.get(sc.get('criterion'))
            if c_name is not None:
                criteria[c_name] = sc.get('score', np.nan)
            j = sc.get('justification')
            if isinstance(j, str) and j.strip():
                justifications.append(j.strip())

    overall = ev.get('overall_rating') or {}
    overall_score = overall.get('final_numeric_score', np.nan)
    overall_summary = overall.get('narrative_summary', '')

    review_text = "\n\n".join(justifications + ([overall_summary] if overall_summary else []))

    row = {
        'proposal_id': r.get('proposal_id'),
        'title': r.get('title'),
        'title_norm': normalize_title(r.get('title', '')),
        'author': r.get('author'),
        'evaluator': r.get('evaluator'),
        'overall_score': overall_score,
        'review_text': review_text,
    }
    row.update(criteria)
    rows.append(row)

ai_df = pd.DataFrame(rows)
ai_df['proposal_uid'] = ai_df['author'].astype(str) + '::' + ai_df['proposal_id'].astype(str)

for c in ['overall_score'] + CRITERIA_ORDER:
    ai_df[c] = pd.to_numeric(ai_df[c], errors='coerce')

print('AI reviews rows:', len(ai_df))
print('Authors:', sorted(ai_df['author'].dropna().unique()))
print('Evaluators:', sorted(ai_df['evaluator'].dropna().unique()))
ai_df.head(3)


AI reviews rows: 276
Authors: ['claude-opus-4-5', 'gemini-3-pro-preview', 'gpt-5.2', 'human-y1', 'human-y2']
Evaluators: ['claude-opus-4-5', 'gemini-3-pro-preview', 'gpt-5.2']


,proposal_id,title,title_norm,author,evaluator,overall_score,review_text,Relevance_to_Emergent_Phenomena,Novelty_and_Significance,Rigor_of_Approach,Scope_and_Timeline,Synthesis_Focus,Data_Identification,Open_Science_Commitment,proposal_uid
0,P1137f1830a,CYTOKINETIC BOTTLENECKS OF HEAT WAVES,cytokinetic bottlenecks of heat waves,human-y1,gpt-5.2,3.2,The proposal targets cytokinesis as a mesoscal...,4,4,3,3,5,3,2,human-y1::P1137f1830a
1,Pcc4e4d7ee4,Elucidating genome structure-function relation...,elucidating genome structure function relation...,human-y1,gpt-5.2,2.7,The proposal targets mesoscale emergence by li...,4,4,3,2,4,2,2,human-y1::Pcc4e4d7ee4
2,P832af59140,MaiTool - LLM-powered bioinformatics tools for...,maitool llm powered bioinformatics tools for m...,human-y1,gpt-5.2,2.5,The proposal focuses on building an AI-enabled...,2,3,2,2,4,3,2,human-y1::P832af59140


## 4) Load and flatten human Y1 expert reviews


In [31]:
human_raw = pd.read_excel(HUMAN_Y1_REVIEWS_PATH)

human_df = human_raw.copy()
human_df['author'] = 'human-y1'
human_df['evaluator'] = human_df['reviewer_id'].astype(str).map(lambda x: f'human-reviewer-{x}')
human_df['title_norm'] = human_df['title'].map(normalize_title)

for col in HUMAN_COL_MAP.keys():
    human_df[col] = pd.to_numeric(human_df[col], errors='coerce')

human_df['overall_score'] = pd.to_numeric(human_df['overall_rating_score'], errors='coerce')
human_df['overall_score'] = human_df['overall_score'].fillna(
    human_df[list(HUMAN_COL_MAP.keys())].mean(axis=1, skipna=True)
)

for c in CRITERIA_ORDER:
    human_df[c] = np.nan

for src_col, targets in HUMAN_COL_MAP.items():
    for t in targets:
        human_df[t] = human_df[src_col]

human_text_cols = [
    'scientific_merit_and_innovation_justification',
    'feasibility_justification',
    'data_sources_and_limitations_justification',
    'open_science_compliance_justification',
    'overall_rating_summary',
]

def merge_text(row):
    parts = [str(row[c]).strip() for c in human_text_cols if isinstance(row[c], str) and row[c].strip()]
    return "\n\n".join(parts)

human_df['review_text'] = human_df.apply(merge_text, axis=1)

human_df = human_df[[
    'id', 'title', 'title_norm', 'author', 'evaluator', 'overall_score', 'review_text', *CRITERIA_ORDER
]].rename(columns={'id': 'proposal_id'})

human_df['proposal_id'] = human_df['proposal_id'].astype(str)
human_df['proposal_uid'] = 'human-y1::' + human_df['proposal_id']

print('Human Y1 rows:', len(human_df))
print('Unique proposals:', human_df['proposal_id'].nunique())
human_df.head(3)


Human Y1 rows: 47
Unique proposals: 12


,proposal_id,title,title_norm,author,evaluator,overall_score,review_text,Relevance_to_Emergent_Phenomena,Novelty_and_Significance,Rigor_of_Approach,Scope_and_Timeline,Synthesis_Focus,Data_Identification,Open_Science_Commitment,proposal_uid
0,1,CYTOKINETIC BOTTLENECKS OF HEAT WAVES,cytokinetic bottlenecks of heat waves,human-y1,human-reviewer-1,NaN,Team leaders identify cytokinesis as a major c...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,human-y1::1
1,1,CYTOKINETIC BOTTLENECKS OF HEAT WAVES,cytokinetic bottlenecks of heat waves,human-y1,human-reviewer-2,5.0,Relevance to Emergent Phenomena: \nThe proposa...,5.0,5.0,5.0,4.0,5.0,5.0,5.0,human-y1::1
2,1,CYTOKINETIC BOTTLENECKS OF HEAT WAVES,cytokinetic bottlenecks of heat waves,human-y1,human-reviewer-3,5.0,This proposal will adress an important and eme...,4.0,4.0,4.0,4.0,5.0,5.0,3.0,human-y1::1


## 5) Proposal matching diagnostics (exact + fuzzy fallback)

This fixes the prior issue where title-exact matching dropped proposals.


In [32]:
ai_y1 = ai_df[ai_df['author'] == 'human-y1'].copy()

human_props = (
    human_df[['proposal_id', 'title', 'title_norm']]
    .drop_duplicates(subset=['proposal_id'])
    .rename(columns={
        'proposal_id': 'human_proposal_id',
        'title': 'human_title',
        'title_norm': 'human_title_norm',
    })
)

ai_props = (
    ai_y1[['proposal_id', 'title', 'title_norm']]
    .drop_duplicates(subset=['proposal_id'])
    .rename(columns={
        'proposal_id': 'ai_proposal_id',
        'title': 'ai_title',
        'title_norm': 'ai_title_norm',
    })
)

mapping_df, mapping_diag, h_unmatched, a_unmatched = build_one_to_one_title_mapping(
    human_props,
    ai_props,
    fuzzy_threshold=0.70,
)

mapping_df = mapping_df.sort_values('human_proposal_id').reset_index(drop=True)
mapping_df['proposal_key'] = [f'Y1_{i+1:02d}' for i in range(len(mapping_df))]

print('Mapping diagnostics:', mapping_diag)
print('Unmatched human proposal_ids:', h_unmatched)
print('Unmatched ai proposal_ids:', a_unmatched)

mapping_df[['proposal_key', 'human_proposal_id', 'ai_proposal_id', 'match_method', 'similarity', 'human_title', 'ai_title']]


Mapping diagnostics: {'n_human_proposals': 12, 'n_ai_proposals': 12, 'n_matched': 12, 'n_exact': 9, 'n_fuzzy': 3, 'n_unmatched_human': 0, 'n_unmatched_ai': 0}
Unmatched human proposal_ids: []
Unmatched ai proposal_ids: []


,proposal_key,human_proposal_id,ai_proposal_id,match_method,similarity,human_title,ai_title
0,Y1_01,1,P1137f1830a,exact,1.000000,CYTOKINETIC BOTTLENECKS OF HEAT WAVES,CYTOKINETIC BOTTLENECKS OF HEAT WAVES
1,Y1_02,10,P2e4d6a4dc3,exact,1.000000,Transposable elements and the emergence of gen...,Transposable elements and the emergence of gen...
2,Y1_03,11,P6d1758976c,exact,1.000000,Searching the Crosslinking Mass Spectrometry ...,Searching the crosslinking mass spectrometry u...
3,Y1_04,12,Pb57c0ae100,fuzzy,0.910877,Intelligent metadata compilation to enhance th...,Intelligent metadata compilation to enhance th...
4,Y1_05,2,Pcc4e4d7ee4,exact,1.000000,Elucidating genome structure-function relation...,Elucidating genome structure-function relation...
5,Y1_06,3,P832af59140,exact,1.000000,MaiTool - LLM-powered bioinformatics tools for...,MaiTool - LLM-powered bioinformatics tools for...
6,Y1_07,4,P947d724043,exact,1.000000,Synthesizing gene expression data to create an...,Synthesizing gene expression data to create an...
7,Y1_08,5,P413efbcd14,exact,1.000000,Energetic Origins of Connectivity within Prote...,Energetic Origins of Connectivity within Prote...
8,Y1_09,6,P7333716d1d,exact,1.000000,Elucidating emergent structures in cellular RN...,Elucidating emergent structures in cellular RN...
9,Y1_10,7,P2e17f59f45,fuzzy,0.857375,Elucidating the Role of Disordered Proteins in...,Oceans of Disorder: Elucidating the Role of Di...


## 6) Build matched review sets, embeddings, and pair table (single source of truth)


In [33]:
MODEL_NAME = 'michiyasunaga/BioLinkBERT-large'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()


def embed_texts(texts, batch_size=8, max_len=512):
    all_emb = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            tok = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors='pt'
            )
            tok = {k: v.to(device) for k, v in tok.items()}
            out = model(**tok)

            attn = tok['attention_mask'].unsqueeze(-1)
            masked = out.last_hidden_state * attn
            denom = attn.sum(dim=1).clamp(min=1)
            emb = masked.sum(dim=1) / denom
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            all_emb.append(emb.cpu().numpy())
    return np.vstack(all_emb)


h = human_df.merge(
    mapping_df[['human_proposal_id', 'proposal_key', 'match_method']],
    left_on='proposal_id',
    right_on='human_proposal_id',
    how='inner',
).copy()

a = ai_y1.merge(
    mapping_df[['ai_proposal_id', 'proposal_key', 'match_method']],
    left_on='proposal_id',
    right_on='ai_proposal_id',
    how='inner',
).copy()

h = h.reset_index(drop=True)
a = a.reset_index(drop=True)
h['review_uid'] = [f'H_{i:04d}' for i in range(len(h))]
a['review_uid'] = [f'A_{i:04d}' for i in range(len(a))]

h_emb = embed_texts(h['review_text'].fillna('').tolist())
a_emb = embed_texts(a['review_text'].fillna('').tolist())

h['embedding'] = list(h_emb)
a['embedding'] = list(a_emb)

for df in [h, a]:
    pol = [TextBlob(t).sentiment.polarity for t in df['review_text'].fillna('')]
    df['polarity'] = pol
    df['sent_label'] = [sentiment_label(p) for p in pol]

records = []
for pk in sorted(set(h['proposal_key']).intersection(set(a['proposal_key']))):
    hg = h[h['proposal_key'] == pk]
    ag = a[a['proposal_key'] == pk]

    for _, hr in hg.iterrows():
        for _, ar in ag.iterrows():
            cos = float(cosine_similarity([hr['embedding']], [ar['embedding']])[0, 0])
            align = sentiment_alignment(hr['polarity'], ar['polarity'])
            cag = categorical_agreement(hr['sent_label'], ar['sent_label'])
            records.append({
                'pair_type': 'human-ai',
                'proposal_key': pk,
                'match_method': hr['match_method'],
                'human_reviewer': hr['evaluator'],
                'ai_model': ar['evaluator'],
                'cosine_similarity': cos,
                'sentiment_alignment': align,
                'categorical_agreement': cag,
                'categorical_agreement_num': {'disagree': 0, 'partial': 1, 'agree': 2}[cag],
            })

    for (_, a1), (_, a2) in itertools.combinations(ag.iterrows(), 2):
        cos = float(cosine_similarity([a1['embedding']], [a2['embedding']])[0, 0])
        align = sentiment_alignment(a1['polarity'], a2['polarity'])
        cag = categorical_agreement(a1['sent_label'], a2['sent_label'])
        m1 = a1['evaluator']
        m2 = a2['evaluator']
        records.append({
            'pair_type': 'ai-ai',
            'proposal_key': pk,
            'match_method': a1['match_method'],
            'ai_model_1': m1,
            'ai_model_2': m2,
            'ai_model_pair': ' vs '.join(sorted([str(m1), str(m2)])),
            'cosine_similarity': cos,
            'sentiment_alignment': align,
            'categorical_agreement': cag,
            'categorical_agreement_num': {'disagree': 0, 'partial': 1, 'agree': 2}[cag],
        })

    for (_, h1), (_, h2) in itertools.combinations(hg.iterrows(), 2):
        cos = float(cosine_similarity([h1['embedding']], [h2['embedding']])[0, 0])
        align = sentiment_alignment(h1['polarity'], h2['polarity'])
        cag = categorical_agreement(h1['sent_label'], h2['sent_label'])
        records.append({
            'pair_type': 'human-human',
            'proposal_key': pk,
            'match_method': h1['match_method'],
            'human_reviewer_1': h1['evaluator'],
            'human_reviewer_2': h2['evaluator'],
            'cosine_similarity': cos,
            'sentiment_alignment': align,
            'categorical_agreement': cag,
            'categorical_agreement_num': {'disagree': 0, 'partial': 1, 'agree': 2}[cag],
        })

pair_df = pd.DataFrame(records)

print('Matched proposals:', pair_df['proposal_key'].nunique())
print('Pair counts by type:')
print(pair_df['pair_type'].value_counts())
print('Human-AI pairs by AI model:')
print(pair_df[pair_df['pair_type'] == 'human-ai']['ai_model'].value_counts())
print('AI-AI pairs by model pair:')
print(pair_df[pair_df['pair_type'] == 'ai-ai']['ai_model_pair'].value_counts())

pair_df.head()


Matched proposals: 12
Pair counts by type:
pair_type
human-ai       141
human-human     70
ai-ai           36
Name: count, dtype: int64
Human-AI pairs by AI model:
ai_model
gpt-5.2                 47
gemini-3-pro-preview    47
claude-opus-4-5         47
Name: count, dtype: int64
AI-AI pairs by model pair:
ai_model_pair
gemini-3-pro-preview vs gpt-5.2            12
claude-opus-4-5 vs gpt-5.2                 12
claude-opus-4-5 vs gemini-3-pro-preview    12
Name: count, dtype: int64


,pair_type,proposal_key,match_method,human_reviewer,ai_model,cosine_similarity,sentiment_alignment,categorical_agreement,categorical_agreement_num,ai_model_1,ai_model_2,ai_model_pair,human_reviewer_1,human_reviewer_2
0,human-ai,Y1_01,exact,human-reviewer-1,gpt-5.2,0.953822,0.953216,partial,1,NaN,NaN,NaN,NaN,NaN
1,human-ai,Y1_01,exact,human-reviewer-1,gemini-3-pro-preview,0.966595,0.937445,partial,1,NaN,NaN,NaN,NaN,NaN
2,human-ai,Y1_01,exact,human-reviewer-1,claude-opus-4-5,0.976407,0.940599,partial,1,NaN,NaN,NaN,NaN,NaN
3,human-ai,Y1_01,exact,human-reviewer-2,gpt-5.2,0.953408,0.988881,agree,2,NaN,NaN,NaN,NaN,NaN
4,human-ai,Y1_01,exact,human-reviewer-2,gemini-3-pro-preview,0.974722,0.995349,agree,2,NaN,NaN,NaN,NaN,NaN


## 7) Pair count checks (expected vs observed)


In [34]:
def _nC2(n: int) -> int:
    n = int(n)
    return (n * (n - 1)) // 2 if n >= 2 else 0

unique_pks = sorted(set(h['proposal_key']).intersection(set(a['proposal_key'])))
human_counts = h.groupby('proposal_key').size().reindex(unique_pks, fill_value=0).rename('n_human_reviews')
ai_counts_total = a.groupby('proposal_key').size().reindex(unique_pks, fill_value=0).rename('n_ai_reviews_total')
ai_counts_by_model = (
    a.pivot_table(index='proposal_key', columns='evaluator', values='review_uid', aggfunc='size', fill_value=0)
    .reindex(unique_pks, fill_value=0)
)

counts_df = pd.concat([human_counts, ai_counts_total, ai_counts_by_model], axis=1).fillna(0).astype(int)
display(counts_df)

expected_human_ai_total = int((counts_df['n_human_reviews'] * counts_df['n_ai_reviews_total']).sum())
expected_human_human_total = int(counts_df['n_human_reviews'].map(_nC2).sum())
expected_ai_ai_total = int(counts_df['n_ai_reviews_total'].map(_nC2).sum())

observed_totals = pair_df['pair_type'].value_counts().to_dict()

print('Expected totals:', {
    'human-ai': expected_human_ai_total,
    'human-human': expected_human_human_total,
    'ai-ai': expected_ai_ai_total,
})
print('Observed totals:', observed_totals)


,n_human_reviews,n_ai_reviews_total,claude-opus-4-5,gemini-3-pro-preview,gpt-5.2
proposal_key,,,,,
Y1_01,4,3,1,1,1
Y1_02,4,3,1,1,1
Y1_03,4,3,1,1,1
Y1_04,4,3,1,1,1
Y1_05,3,3,1,1,1
Y1_06,4,3,1,1,1
Y1_07,4,3,1,1,1
Y1_08,4,3,1,1,1
Y1_09,4,3,1,1,1


Expected totals: {'human-ai': 141, 'human-human': 70, 'ai-ai': 36}
Observed totals: {'human-ai': 141, 'human-human': 70, 'ai-ai': 36}


## 8) Similarity proxy stats (proposal-level, model-aware, FDR-corrected)

**Purpose**
This section evaluates whether AI review text behaves like a reliable proxy for human expert review text quality judgments.

**How this is done**
1. Aggregate pairwise metrics to proposal-level means (human-human, human-AI, AI-AI) to reduce pseudo-replication.
2. Compare groups with Mann-Whitney U and Cliff's delta.
3. Add paired Wilcoxon tests on shared proposals as a within-proposal robustness check.
4. Apply Benjamini-Hochberg FDR correction within each metric family.
5. Break down results by AI model (`human-ai`) and AI model pair (`ai-ai`).

**Why this method is appropriate**
- Proposal-level aggregation respects the unit of inference (proposal) better than raw pair rows.
- Mann-Whitney + Cliff's delta is robust for non-normal small-sample settings.
- Wilcoxon complements this by exploiting paired structure over shared proposals.
- FDR correction controls false discoveries across multiple related tests.


In [ ]:
# Purpose:
# Summarize similarity metrics at proposal-level and test whether human-AI / AI-AI
# comparisons differ from human-human baseline.
#
# Method:
# 1) Aggregate pairwise rows to one value per proposal per comparison type.
# 2) Run Mann-Whitney U + Cliff's delta for distributional differences.
# 3) Add paired Wilcoxon tests across shared proposals.
# 4) Apply BH-FDR within metric families.

SIM_METRICS = ['cosine_similarity', 'sentiment_alignment', 'categorical_agreement_num']

human_human_prop = (
    pair_df[pair_df['pair_type'] == 'human-human']
    .groupby('proposal_key')[SIM_METRICS]
    .mean()
    .reset_index()
)
human_ai_prop = (
    pair_df[pair_df['pair_type'] == 'human-ai']
    .groupby('proposal_key')[SIM_METRICS]
    .mean()
    .reset_index()
)
ai_ai_prop = (
    pair_df[pair_df['pair_type'] == 'ai-ai']
    .groupby('proposal_key')[SIM_METRICS]
    .mean()
    .reset_index()
)

overall_tests = []
for m in SIM_METRICS:
    for g1_name, g1_df, g2_name, g2_df in [
        ('human-ai', human_ai_prop, 'human-human', human_human_prop),
        ('ai-ai', ai_ai_prop, 'human-human', human_human_prop),
        ('ai-ai', ai_ai_prop, 'human-ai', human_ai_prop),
    ]:
        out = mannwhitney_summary(g1_df[m], g2_df[m], metric_name=m)
        out['group1'] = g1_name
        out['group2'] = g2_name

        merged = g1_df[['proposal_key', m]].merge(g2_df[['proposal_key', m]], on='proposal_key', suffixes=('_1', '_2'))
        try:
            w_stat, w_p = wilcoxon(merged[f'{m}_1'], merged[f'{m}_2'])
        except Exception:
            w_stat, w_p = np.nan, np.nan
        out['wilcoxon_stat'] = w_stat
        out['wilcoxon_p_value'] = w_p
        out['n_paired_proposals'] = len(merged)

        overall_tests.append(out)

sim_stats_overall_df = pd.DataFrame(overall_tests)
sim_stats_overall_df = add_bh_fdr(sim_stats_overall_df, p_col='p_value', group_cols=['metric'])

human_ai_by_model_prop = (
    pair_df[pair_df['pair_type'] == 'human-ai']
    .groupby(['proposal_key', 'ai_model'])[SIM_METRICS]
    .mean()
    .reset_index()
)

rows = []
for m in SIM_METRICS:
    for model in sorted(human_ai_by_model_prop['ai_model'].dropna().unique()):
        sub = human_ai_by_model_prop[human_ai_by_model_prop['ai_model'] == model]
        out = mannwhitney_summary(sub[m], human_human_prop[m], metric_name=m)
        out['group1'] = f'human-ai::{model}'
        out['group2'] = 'human-human'
        out['ai_model'] = model
        rows.append(out)

sim_stats_hai_by_model_df = pd.DataFrame(rows)
sim_stats_hai_by_model_df = add_bh_fdr(sim_stats_hai_by_model_df, p_col='p_value', group_cols=['metric'])

ai_ai_by_pair_prop = (
    pair_df[pair_df['pair_type'] == 'ai-ai']
    .groupby(['proposal_key', 'ai_model_pair'])[SIM_METRICS]
    .mean()
    .reset_index()
)

rows = []
for m in SIM_METRICS:
    for mp in sorted(ai_ai_by_pair_prop['ai_model_pair'].dropna().unique()):
        sub = ai_ai_by_pair_prop[ai_ai_by_pair_prop['ai_model_pair'] == mp]
        out = mannwhitney_summary(sub[m], human_human_prop[m], metric_name=m)
        out['group1'] = f'ai-ai::{mp}'
        out['group2'] = 'human-human'
        out['ai_model_pair'] = mp
        rows.append(out)

sim_stats_aiai_by_pair_df = pd.DataFrame(rows)
sim_stats_aiai_by_pair_df = add_bh_fdr(sim_stats_aiai_by_pair_df, p_col='p_value', group_cols=['metric'])

print('Overall pair-type stats:')
display(sim_stats_overall_df)
print('Human-AI by model stats:')
display(sim_stats_hai_by_model_df)
print('AI-AI by model-pair stats:')
display(sim_stats_aiai_by_pair_df)


### Interpretation: Similarity Proxy Statistics

- **Primary columns to use:** `q_value`, `cliffs_delta`, and `delta_magnitude`.
- If `q_value < 0.05`, the comparison is statistically distinguishable after multiple-testing correction.
- Use Cliff's delta sign/direction to report *which group tends to be higher* and magnitude labels for practical importance.
- Compare `p_value` vs `wilcoxon_p_value`: if both agree, inference is more robust; if they diverge, emphasize uncertainty and sample dependence.


In [ ]:
# Purpose:
# Visualize proposal-level similarity distributions overall and by model decomposition.
#
# Why helpful:
# Plots reveal overlap, spread, and outliers that are not obvious from p-values alone.

stacked = []
for label, df_ in [('human-human', human_human_prop), ('human-ai', human_ai_prop), ('ai-ai', ai_ai_prop)]:
    tmp = df_.copy()
    tmp['pair_type'] = label
    stacked.append(tmp)
plot_prop = pd.concat(stacked, ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
pt_order = ['human-human', 'human-ai', 'ai-ai']
pt_palette = {'human-human': colors['Human'], 'human-ai': '#888888', 'ai-ai': '#2E8B57'}

for ax, metric, title in zip(
    axes,
    SIM_METRICS,
    ['Cosine Similarity', 'Sentiment Alignment', 'Categorical Agreement (0/1/2)']
):
    sns.boxplot(data=plot_prop, x='pair_type', y=metric, order=pt_order, ax=ax, palette=pt_palette)
    sns.stripplot(data=plot_prop, x='pair_type', y=metric, order=pt_order, ax=ax, color='black', alpha=0.35, size=4)
    ax.set_title(title)
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'quality_similarity_proxy_boxplots_proposal_level.png', dpi=200, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
model_order = sorted(human_ai_by_model_prop['ai_model'].dropna().unique())
model_palette = {m: colors.get(m, '#999999') for m in model_order}
for ax, metric, title in zip(axes, SIM_METRICS, ['Cosine Similarity', 'Sentiment Alignment', 'Categorical Agreement (0/1/2)']):
    sns.boxplot(data=human_ai_by_model_prop, x='ai_model', y=metric, order=model_order, ax=ax, palette=model_palette)
    sns.stripplot(data=human_ai_by_model_prop, x='ai_model', y=metric, order=model_order, ax=ax, color='black', alpha=0.25, size=3)
    ax.set_title(f'Human-AI by model: {title}')
    ax.tick_params(axis='x', rotation=20)
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'quality_similarity_human_ai_by_model_proposal_level.png', dpi=200, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
pair_order = sorted(ai_ai_by_pair_prop['ai_model_pair'].dropna().unique())
for ax, metric, title in zip(axes, SIM_METRICS, ['Cosine Similarity', 'Sentiment Alignment', 'Categorical Agreement (0/1/2)']):
    sns.boxplot(data=ai_ai_by_pair_prop, x='ai_model_pair', y=metric, order=pair_order, ax=ax, color='#2E8B57')
    sns.stripplot(data=ai_ai_by_pair_prop, x='ai_model_pair', y=metric, order=pair_order, ax=ax, color='black', alpha=0.25, size=3)
    ax.set_title(f'AI-AI by model pair: {title}')
    ax.tick_params(axis='x', rotation=20)
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'quality_similarity_ai_ai_by_model_pair_proposal_level.png', dpi=200, bbox_inches='tight')
plt.show()


### Interpretation: Similarity Plots

- **Human-human** provides the inter-expert baseline.
- If **human-AI** distribution overlaps strongly with human-human, AI reviews behave more like human proxy ratings.
- If **AI-AI** is much tighter/higher than human-human, AI reviewers may be internally consistent but not necessarily human-aligned.
- In model-specific plots, identify which AI model drives deviation from human baseline.


## 9) Matching sensitivity analysis (exact-only vs exact+fuzzy)

**Purpose**
Check whether conclusions depend on title-matching strategy.

**How this is done**
- Recompute the core similarity comparisons twice:
  1. `exact_only` matched proposals
  2. `exact_plus_fuzzy` matched proposals
- Compare effect sizes and significance patterns.

**Why this method is appropriate**
Matching uncertainty is a key source of analytic fragility. Sensitivity analysis shows whether inferences are robust to reasonable matching choices.


In [ ]:
# Purpose:
# Recompute the same core similarity tests under alternative matching strategies
# to check robustness of statistical conclusions.

def compute_similarity_overall_from_pair_df(df_):
    out_rows = []
    hh = df_[df_['pair_type'] == 'human-human'].groupby('proposal_key')[SIM_METRICS].mean().reset_index()
    ha = df_[df_['pair_type'] == 'human-ai'].groupby('proposal_key')[SIM_METRICS].mean().reset_index()
    aa = df_[df_['pair_type'] == 'ai-ai'].groupby('proposal_key')[SIM_METRICS].mean().reset_index()

    for m in SIM_METRICS:
        for g1_name, g1_df, g2_name, g2_df in [
            ('human-ai', ha, 'human-human', hh),
            ('ai-ai', aa, 'human-human', hh),
            ('ai-ai', aa, 'human-ai', ha),
        ]:
            row = mannwhitney_summary(g1_df[m], g2_df[m], metric_name=m)
            row['comparison'] = f'{g1_name} vs {g2_name}'
            out_rows.append(row)
    return pd.DataFrame(out_rows)

exact_keys = set(mapping_df.loc[mapping_df['match_method'] == 'exact', 'proposal_key'])
all_keys = set(mapping_df['proposal_key'])

pair_exact = pair_df[pair_df['proposal_key'].isin(exact_keys)].copy()
pair_full = pair_df[pair_df['proposal_key'].isin(all_keys)].copy()

sens_exact = compute_similarity_overall_from_pair_df(pair_exact)
sens_exact['strategy'] = 'exact_only'
sens_exact['n_proposals'] = len(exact_keys)

sens_full = compute_similarity_overall_from_pair_df(pair_full)
sens_full['strategy'] = 'exact_plus_fuzzy'
sens_full['n_proposals'] = len(all_keys)

matching_sensitivity_df = pd.concat([sens_exact, sens_full], ignore_index=True)
matching_sensitivity_df = add_bh_fdr(matching_sensitivity_df, p_col='p_value', group_cols=['strategy', 'metric'])
matching_sensitivity_df


### Interpretation: Matching Sensitivity

- Compare `exact_only` vs `exact_plus_fuzzy` rows.
- If effect directions and significance are stable across strategies, conclusions are robust to matching uncertainty.
- If results flip, report that the proxy conclusion is sensitive to linkage assumptions.


## 10) Proposal-quality analysis dataset (proposal-level means, no duplicated human-all rows)

**Purpose**
Create the main proposal-level scoring dataset for quality comparisons.

**How this is done**
- Average scores across evaluators within each proposal.
- Keep true base groups (`human-y1`, `human-y2`, and each AI author model).
- Construct `human-all` only analytically (not by duplicating rows in source data).

**Why this method is appropriate**
- Proposal-level summaries align with the research question: proposal quality, not individual review idiosyncrasy.
- Avoiding duplicated rows prevents biased evaluator-level analyses and inflated sample counts.


In [ ]:
# Purpose:
# Build proposal-level quality dataset and summary stats without contaminating source
# data via duplicated synthetic groups.

QUALITY_METRICS = ['overall_score'] + CRITERIA_ORDER

proposal_scores = (
    ai_df.groupby(['author', 'proposal_id', 'proposal_uid'])[QUALITY_METRICS]
    .mean()
    .reset_index()
)

base_groups = ['human-y1', 'human-y2', 'claude-opus-4-5', 'gemini-3-pro-preview', 'gpt-5.2']
proposal_scores = proposal_scores[proposal_scores['author'].isin(base_groups)].copy()


def get_group_values(df_, group_name, metric):
    if group_name == 'human-all':
        return df_[df_['author'].isin(['human-y1', 'human-y2'])][metric].dropna().values
    return df_[df_['author'] == group_name][metric].dropna().values


keep_groups = ['human-y1', 'human-y2', 'human-all', 'claude-opus-4-5', 'gemini-3-pro-preview', 'gpt-5.2']

summary_rows = []
for g in keep_groups:
    vals = proposal_scores[proposal_scores['author'].isin(['human-y1', 'human-y2'])]['overall_score'] if g == 'human-all' else proposal_scores[proposal_scores['author'] == g]['overall_score']
    summary_rows.append({
        'group': g,
        'n_proposals': int(vals.notna().sum()),
        'overall_mean': float(vals.mean()),
        'overall_median': float(vals.median()),
        'overall_std': float(vals.std()),
    })

summary_overall_df = pd.DataFrame(summary_rows)
summary_overall_df


### Interpretation: Proposal-Level Summary Dataset

- Check `n_proposals` per group first; unequal group sizes affect power.
- `overall_mean`/`median` provide central tendency; `overall_std` indicates heterogeneity.
- Use this table to contextualize later pairwise tests (e.g., large variance can weaken significance despite mean gaps).


In [ ]:
# Purpose:
# Visual diagnostics for quality distributions and criterion profiles.
#
# Note:
# 'human-all' is generated only for comparative plotting convenience.

plot_frames = []
for g in keep_groups:
    if g == 'human-all':
        tmp = proposal_scores[proposal_scores['author'].isin(['human-y1', 'human-y2'])].copy()
    else:
        tmp = proposal_scores[proposal_scores['author'] == g].copy()
    tmp['author_group'] = g
    plot_frames.append(tmp)
plot_df = pd.concat(plot_frames, ignore_index=True)

plt.figure(figsize=(12, 7))
for g in keep_groups:
    s = plot_df.loc[plot_df['author_group'] == g, 'overall_score'].dropna()
    if len(s) == 0:
        continue
    color = colors['Human'] if 'human' in g else colors.get(g, '#999999')
    plt.hist(s, bins=np.arange(1, 5.6, 0.25), alpha=0.35, label=g, color=color, density=True)
plt.xlabel('Overall score')
plt.ylabel('Density')
plt.title('Overall proposal-level score distributions')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'quality_overall_histograms_proposal_level.png', dpi=200, bbox_inches='tight')
plt.show()

plt.figure(figsize=(13, 6))
palette = {g: (colors['Human'] if 'human' in g else colors.get(g, '#999999')) for g in keep_groups}
sns.boxplot(data=plot_df, x='author_group', y='overall_score', order=keep_groups, palette=palette)
sns.stripplot(data=plot_df, x='author_group', y='overall_score', order=keep_groups, color='black', alpha=0.25, size=3)
plt.xticks(rotation=20, ha='right')
plt.title('Overall score by author group (proposal-level)')
plt.xlabel('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'quality_overall_boxplot_proposal_level.png', dpi=200, bbox_inches='tight')
plt.show()

radar_groups = ['human-all', 'claude-opus-4-5', 'gemini-3-pro-preview', 'gpt-5.2']

# Short display labels matching the reference style
radar_labels = {
    'Relevance_to_Emergent_Phenomena': 'Relevance to\nEmergent Phenomena',
    'Novelty_and_Significance':        'Novelty &\nSignificance',
    'Rigor_of_Approach':               'Rigor of\nApproach',
    'Scope_and_Timeline':              'Scope & Timeline',
    'Synthesis_Focus':                 'Synthesis Focus',
    'Data_Identification':             'Data Identification',
    'Open_Science_Commitment':         'Open Science',
}

# Marker styles per group
radar_markers = {
    'human-all':           'o',
    'claude-opus-4-5':     's',
    'gemini-3-pro-preview':'D',
    'gpt-5.2':             '^',
}

radar_legend_names = {
    'human-all':           'Human (all)',
    'claude-opus-4-5':     'claude-opus-4-5',
    'gemini-3-pro-preview':'gemini-3-pro-preview',
    'gpt-5.2':             'gpt-5.2',
}

N = len(CRITERIA_ORDER)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig = plt.figure(figsize=(10, 10), facecolor='white')
ax = fig.add_subplot(111, polar=True, facecolor='white')

# Light gray gridlines only, no outer spine
ax.set_ylim(1, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels([])
ax.yaxis.grid(True, color='#cccccc', linewidth=0.8, linestyle='-')
ax.xaxis.grid(True, color='#cccccc', linewidth=0.8, linestyle='-')
ax.spines['polar'].set_visible(False)

# Scale numbers 1-4 along the first spoke (rightward horizontal)
for tick_val in [1, 2, 3, 4]:
    ax.text(angles[0], tick_val, str(tick_val),
            ha='center', va='center', fontsize=9, color='#666666')

# Plot each group
for g in radar_groups:
    vals = [np.nanmean(get_group_values(proposal_scores, g, c)) for c in CRITERIA_ORDER]
    vals += vals[:1]
    color = colors['Human'] if 'human' in g else colors.get(g, '#999999')
    marker = radar_markers.get(g, 'o')
    ax.plot(angles, vals, linewidth=2.5, color=color,
            marker=marker, markersize=7, markerfacecolor=color,
            label=radar_legend_names.get(g, g))
    ax.fill(angles, vals, alpha=0.12, color=color)

# Axis spoke labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(
    [radar_labels.get(c, c) for c in CRITERIA_ORDER],
    fontsize=10, color='#222222'
)
ax.tick_params(pad=14)

# Legend outside the chart
ax.legend(loc='upper left', bbox_to_anchor=(1.15, 1.15), frameon=False, fontsize=10)

fig.suptitle('Criterion profiles by author group', fontsize=14, fontweight='bold', y=1.01)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'quality_radar_criteria_proposal_level.png', dpi=200, bbox_inches='tight')
plt.show()


### Interpretation: Quality Distribution Plots

- Histogram/boxplot: inspect central location, spread, and skew across groups.
- Radar chart: identify criteria where human vs AI profiles diverge most.
- Prefer combining visual patterns with statistical tables from Sections 11-12 before drawing conclusions.


## 11) Pairwise quality tests (MW + Cliff's delta + FDR) on proposal-level means

**Purpose**
Identify where quality differences exist between proposal source groups (human cohorts vs AI models), overall and by criterion.

**How this is done**
- Run all pairwise group comparisons for each metric.
- Report Mann-Whitney U, p-value, Cliff's delta, and group means.
- Apply FDR correction within each metric.

**Why this method is appropriate**
- Non-parametric tests are suitable for small and potentially skewed distributions.
- Effect sizes (Cliff's delta) provide magnitude and direction beyond p-values.


In [ ]:
# Purpose:
# Exhaustive pairwise group comparisons for each quality metric, with FDR control.

pairwise_results = []
for metric in QUALITY_METRICS:
    for g1, g2 in itertools.combinations(keep_groups, 2):
        s1 = get_group_values(proposal_scores, g1, metric)
        s2 = get_group_values(proposal_scores, g2, metric)
        out = mannwhitney_summary(s1, s2, metric_name=metric)
        out['group1'] = g1
        out['group2'] = g2
        out['mean_group1'] = np.mean(s1) if len(s1) else np.nan
        out['mean_group2'] = np.mean(s2) if len(s2) else np.nan
        pairwise_results.append(out)

pairwise_df = pd.DataFrame(pairwise_results)
pairwise_df = add_bh_fdr(pairwise_df, p_col='p_value', group_cols=['metric'])

vs_human_all = pairwise_df[(pairwise_df['group1'] == 'human-all') | (pairwise_df['group2'] == 'human-all')].copy()
vs_human_all.sort_values(['metric', 'q_value', 'p_value']).head(40)


### Interpretation: Pairwise Quality Tests

- For each metric, prioritize `q_value` (FDR-adjusted) over raw `p_value`.
- Report both **statistical evidence** (`q_value`) and **practical effect** (`cliffs_delta`, `delta_magnitude`).
- In `vs_human_all`, positive vs negative delta tells whether the comparator tends to score above or below `human-all`.


## 12) Robust inference for key comparisons (bootstrap CI + permutation p)

**Purpose**
Stress-test key human-vs-AI conclusions with distribution-free inference.

**How this is done**
- Bootstrap confidence intervals for mean differences.
- Permutation tests for mean-difference significance.
- FDR correction across permutation p-values.

**Why this method is appropriate**
- Bootstrap CIs quantify uncertainty without strict normality assumptions.
- Permutation tests provide valid small-sample inference under exchangeability.


In [ ]:
# Purpose:
# Robustness checks for key human-vs-AI contrasts using bootstrap CIs and
# permutation p-values.

robust_rows = []
for metric in QUALITY_METRICS:
    x = get_group_values(proposal_scores, 'human-all', metric)
    for g in ['claude-opus-4-5', 'gemini-3-pro-preview', 'gpt-5.2']:
        y = get_group_values(proposal_scores, g, metric)
        mean_diff, ci_lo, ci_hi = bootstrap_mean_diff_ci(x, y, n_boot=2000, ci=0.95, seed=42)
        p_perm = permutation_p_value_mean_diff(x, y, n_perm=5000, seed=42)
        robust_rows.append({
            'metric': metric,
            'comparison': f'human-all minus {g}',
            'n_human_all': len(x),
            'n_other': len(y),
            'mean_diff': mean_diff,
            'bootstrap_ci_lo': ci_lo,
            'bootstrap_ci_hi': ci_hi,
            'permutation_p_value': p_perm,
        })

robust_quality_df = pd.DataFrame(robust_rows)
robust_quality_df = add_bh_fdr(robust_quality_df, p_col='permutation_p_value', group_cols=['metric'], out_col='permutation_q_value')
robust_quality_df.head(20)


### Interpretation: Robust Inference

- `mean_diff` sign indicates direction (`human-all minus model`).
- If bootstrap CI excludes 0 and permutation `q_value < 0.05`, conclusion is robust.
- If MW and robust tests disagree, treat finding as fragile and report as exploratory.


## 13) Evaluator differences (non-duplicated data only)

**Purpose**
Assess whether evaluator models have systematic scoring differences (leniency/stringency).

**How this is done**
- Summarize per-evaluator score distributions.
- Visualize distributions.
- Use Kruskal-Wallis to test overall group differences.

**Why this method is appropriate**
Evaluator drift is a central bias risk in AI-as-judge designs; this provides an explicit diagnostic before interpretation of author-group differences.


In [ ]:
# Purpose:
# Quick evaluator-level descriptive statistics on non-duplicated source data.

eval_stats = ai_df.groupby('evaluator')['overall_score'].agg(['count', 'mean', 'median', 'std']).sort_values('mean', ascending=False)
eval_stats


### Interpretation: Evaluator Descriptives

- Use this table to detect systematic evaluator strictness/leniency before model-comparison claims.
- Large mean/median gaps across evaluators indicate potential judge bias.


In [ ]:
# Purpose:
# Evaluate evaluator distribution differences visually and formally (Kruskal-Wallis).

plt.figure(figsize=(10, 6))
sns.boxplot(data=ai_df, x='evaluator', y='overall_score', palette='Set2')
sns.stripplot(data=ai_df, x='evaluator', y='overall_score', color='black', alpha=0.2, size=2)
plt.title('Overall score by evaluator model (raw ai_df)')
plt.xlabel('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'quality_overall_by_evaluator_clean.png', dpi=200, bbox_inches='tight')
plt.show()

parts = [ai_df.loc[ai_df['evaluator'] == e, 'overall_score'].dropna().values for e in sorted(ai_df['evaluator'].dropna().unique())]
kw_stat, kw_p = kruskal(*parts)
print({'kruskal_stat': kw_stat, 'p_value': kw_p})


### Interpretation: Evaluator Difference Test

- Kruskal-Wallis `p_value < 0.05` indicates at least one evaluator distribution differs.
- This supports modeling evaluator effects explicitly (Sections 14 and 18).


## 14) AI self-preference tests (overall + criterion-level + proposal controls)

**Purpose**
Test whether AI evaluators prefer proposals generated by their own model.

**How this is done**
1. Compare self vs non-self scores at proposal level (overall).
2. Repeat by criterion to localize bias.
3. Fit fixed-effects regression with proposal controls to isolate self-preference net of proposal difficulty.

**Why this method is appropriate**
Self-preference is a known LLM evaluator bias; combining non-parametric and controlled regression analyses strengthens causal interpretability.


In [ ]:
# Purpose:
# Overall self-preference test by evaluator model at proposal level.

ai_authors = ['claude-opus-4-5', 'gemini-3-pro-preview', 'gpt-5.2']

sp = ai_df[ai_df['author'].isin(ai_authors) & ai_df['evaluator'].isin(ai_authors)].copy()
sp_prop = (
    sp.groupby(['evaluator', 'author', 'proposal_id'])[QUALITY_METRICS]
    .mean()
    .reset_index()
)
sp_prop['is_self'] = sp_prop['author'] == sp_prop['evaluator']

self_pref_rows = []
for evaluator in ai_authors:
    sub = sp_prop[sp_prop['evaluator'] == evaluator]
    own = sub[sub['is_self']]['overall_score'].dropna()
    other = sub[~sub['is_self']]['overall_score'].dropna()
    out = mannwhitney_summary(own, other, metric_name='overall_score')
    out['evaluator'] = evaluator
    out['mean_self'] = own.mean() if len(own) else np.nan
    out['mean_other'] = other.mean() if len(other) else np.nan
    self_pref_rows.append(out)

self_pref_df = pd.DataFrame(self_pref_rows)
self_pref_df = add_bh_fdr(self_pref_df, p_col='p_value')
self_pref_df


### Interpretation: Overall Self-Preference

- Compare `mean_self` vs `mean_other` per evaluator.
- `q_value < 0.05` with higher `mean_self` suggests self-favoring bias.
- Opposite direction suggests self-penalization or cross-model preference.


In [ ]:
# Purpose:
# Criterion-level self-preference decomposition to identify where bias concentrates.

sp_long = sp_prop.melt(
    id_vars=['evaluator', 'author', 'proposal_id', 'is_self'],
    value_vars=QUALITY_METRICS,
    var_name='metric',
    value_name='score',
).dropna(subset=['score'])

rows = []
for evaluator in ai_authors:
    sube = sp_long[sp_long['evaluator'] == evaluator]
    for metric in QUALITY_METRICS:
        sm = sube[sube['metric'] == metric]
        own = sm[sm['is_self']]['score']
        other = sm[~sm['is_self']]['score']
        out = mannwhitney_summary(own, other, metric_name=metric)
        out['evaluator'] = evaluator
        out['mean_self'] = own.mean() if len(own) else np.nan
        out['mean_other'] = other.mean() if len(other) else np.nan
        rows.append(out)

self_pref_metric_df = pd.DataFrame(rows)
self_pref_metric_df = add_bh_fdr(self_pref_metric_df, p_col='p_value', group_cols=['evaluator'])
self_pref_metric_df.head(20)


### Interpretation: Criterion-Level Self-Preference

- Identify which criteria show strongest self-preference (`q_value` + large |delta|).
- A bias concentrated in specific criteria is more actionable than uniform global bias.


In [ ]:
# Purpose:
# Fixed-effects regression to isolate self-preference net of proposal effects.

try:
    import statsmodels.formula.api as smf

    fe_df = sp_long.copy()
    fe_df['proposal_uid'] = fe_df['author'].astype(str) + '::' + fe_df['proposal_id'].astype(str)
    fe_df['is_self_num'] = fe_df['is_self'].astype(int)

    model_self = smf.ols(
        'score ~ is_self_num * C(metric) + C(evaluator) + C(author) + C(proposal_uid)',
        data=fe_df
    ).fit(cov_type='HC3')

    print(model_self.summary().tables[1])
except Exception as e:
    print('statsmodels model skipped:', e)


### Interpretation: Fixed-Effects Self-Preference Model

- Focus on coefficient for `is_self_num` and interactions with `C(metric)`.
- Significant positive terms imply self-preference persists after controlling proposal-level difficulty.
- Non-significant terms suggest observed self-preference may be explained by proposal composition.


## 15) Proxy validity vs human experts (shared 4-category rubric)

**Purpose**
Evaluate whether AI reviewer outputs are a good proxy for human expert panel judgments.

**How this is done**
- Harmonize both sides onto shared rubric dimensions.
- Compute error metrics (bias, MAE, RMSE) and agreement metrics (Spearman/Kendall).
- Evaluate ranking agreement on overall score.

**Why this method is appropriate**
Proxy validity requires both calibration (low error) and ordering consistency (agreement in ranking), not just similar means.


In [ ]:
# Purpose:
# Harmonize human and AI evaluations to shared rubric dimensions and build aligned
# proposal-level panels for proxy validity testing.

human_shared = human_raw.copy()
human_shared['proposal_id'] = human_shared['id'].astype(str)
for col in ['scientific_merit_and_innovation_score', 'feasibility_score', 'data_sources_and_limitations_score', 'open_science_compliance_score']:
    human_shared[col] = pd.to_numeric(human_shared[col], errors='coerce')

human_panel = (
    human_shared.groupby('proposal_id')[[
        'scientific_merit_and_innovation_score',
        'feasibility_score',
        'data_sources_and_limitations_score',
        'open_science_compliance_score',
    ]]
    .mean()
    .reset_index()
    .rename(columns={
        'scientific_merit_and_innovation_score': 'Scientific_Merit_and_Innovation',
        'feasibility_score': 'Feasibility',
        'data_sources_and_limitations_score': 'Data_Sources_and_Limitations',
        'open_science_compliance_score': 'Open_Science_Compliance',
    })
)
human_panel['overall_shared'] = human_panel[SHARED_METRICS_4CAT].mean(axis=1)

ai_y1_shared = ai_df[ai_df['author'] == 'human-y1'].copy()
ai_y1_shared['Scientific_Merit_and_Innovation'] = ai_y1_shared[['Relevance_to_Emergent_Phenomena', 'Novelty_and_Significance', 'Rigor_of_Approach']].mean(axis=1)
ai_y1_shared['Feasibility'] = ai_y1_shared['Scope_and_Timeline']
ai_y1_shared['Data_Sources_and_Limitations'] = ai_y1_shared[['Synthesis_Focus', 'Data_Identification']].mean(axis=1)
ai_y1_shared['Open_Science_Compliance'] = ai_y1_shared['Open_Science_Commitment']
ai_y1_shared['overall_shared'] = ai_y1_shared[SHARED_METRICS_4CAT].mean(axis=1)

ai_to_human_id = mapping_df.set_index('ai_proposal_id')['human_proposal_id'].to_dict()
ai_y1_shared['human_proposal_id'] = ai_y1_shared['proposal_id'].map(ai_to_human_id)
ai_y1_shared = ai_y1_shared.dropna(subset=['human_proposal_id']).copy()
ai_y1_shared['human_proposal_id'] = ai_y1_shared['human_proposal_id'].astype(str)

ai_panel = (
    ai_y1_shared.groupby(['evaluator', 'human_proposal_id'])[
        SHARED_METRICS_4CAT + ['overall_shared']
    ]
    .mean()
    .reset_index()
    .rename(columns={'human_proposal_id': 'proposal_id'})
)

print('Human panel proposals:', human_panel['proposal_id'].nunique())
print('AI panel proposals:', ai_panel['proposal_id'].nunique())

human_panel.head(), ai_panel.head()


### Interpretation: Proxy Data Harmonization Check

- Confirm `Human panel proposals` and `AI panel proposals` counts are as expected.
- Mismatch here indicates linkage or missingness issues that must be resolved before interpreting proxy validity.


In [ ]:
# Purpose:
# Compute proxy validity diagnostics:
# - bias / MAE / RMSE (calibration quality)
# - Spearman/Kendall (ordinal agreement)
# - rank agreement for overall selection behavior.

proxy_rows = []
rank_rows = []

for evaluator in sorted(ai_panel['evaluator'].unique()):
    sub_ai = ai_panel[ai_panel['evaluator'] == evaluator].copy()
    merged = sub_ai.merge(human_panel, on='proposal_id', suffixes=('_ai', '_human'))

    for metric in SHARED_METRICS_4CAT + ['overall_shared']:
        a_vals = merged[f'{metric}_ai'].values
        h_vals = merged[f'{metric}_human'].values
        err = a_vals - h_vals

        rho, rho_p = spearmanr(a_vals, h_vals)
        tau, tau_p = kendalltau(a_vals, h_vals)

        proxy_rows.append({
            'evaluator': evaluator,
            'metric': metric,
            'n_proposals': len(merged),
            'bias_mean_error': np.mean(err),
            'mae': np.mean(np.abs(err)),
            'rmse': np.sqrt(np.mean(err ** 2)),
            'spearman_rho': rho,
            'spearman_p': rho_p,
            'kendall_tau': tau,
            'kendall_p': tau_p,
        })

    h_rank = merged['overall_shared_human'].rank(ascending=False)
    a_rank = merged['overall_shared_ai'].rank(ascending=False)
    rrho, rrho_p = spearmanr(h_rank, a_rank)
    rtau, rtau_p = kendalltau(h_rank, a_rank)
    rank_rows.append({
        'evaluator': evaluator,
        'spearman_rank_rho': rrho,
        'spearman_rank_p': rrho_p,
        'kendall_rank_tau': rtau,
        'kendall_rank_p': rtau_p,
        'n_proposals': len(merged),
    })

proxy_validity_df = pd.DataFrame(proxy_rows)
proxy_validity_df = add_bh_fdr(proxy_validity_df, p_col='spearman_p', group_cols=['metric'], out_col='spearman_q')
proxy_validity_df = add_bh_fdr(proxy_validity_df, p_col='kendall_p', group_cols=['metric'], out_col='kendall_q')

rank_agreement_df = pd.DataFrame(rank_rows)
rank_agreement_df = add_bh_fdr(rank_agreement_df, p_col='spearman_rank_p', out_col='spearman_rank_q')
rank_agreement_df = add_bh_fdr(rank_agreement_df, p_col='kendall_rank_p', out_col='kendall_rank_q')

print('Proxy validity table:')
display(proxy_validity_df)
print('Rank agreement table:')
display(rank_agreement_df)


### Interpretation: Proxy Validity Results

- **Bias near 0 + low MAE/RMSE** indicates better calibration to human experts.
- **High Spearman/Kendall** indicates stronger ordering agreement with human panel.
- Use rank table to assess whether AI and human reviewers would select similar top proposals.
- If calibration is poor but rank agreement is high, AI may be useful for triage but not absolute scoring.


## 16) Inter-rater agreement metrics (ICC + optional Krippendorff)

**Purpose**
Quantify reliability across raters in a common framework.

**How this is done**
- ICC(2,1): reliability of single ratings.
- ICC(2,k): reliability of average ratings across raters.
- Optional Krippendorff alpha (ordinal) for mixed-rater robustness with missingness tolerance.

**Why this method is appropriate**
These are standard reliability metrics in review studies and directly address whether ratings are consistently reproducible.


In [ ]:
# Purpose:
# Compute ICC reliability metrics across the combined rater panel.

icc_rows = []
for metric in SHARED_METRICS_4CAT + ['overall_shared']:
    panel = human_panel[['proposal_id', metric]].rename(columns={metric: 'human_panel'})

    for evaluator in sorted(ai_panel['evaluator'].unique()):
        sub = ai_panel[ai_panel['evaluator'] == evaluator][['proposal_id', metric]].rename(columns={metric: evaluator})
        panel = panel.merge(sub, on='proposal_id', how='inner')

    rater_cols = ['human_panel'] + sorted(ai_panel['evaluator'].unique().tolist())
    mat = panel[rater_cols].to_numpy(dtype=float)
    icc21, icc2k = icc2_1_2k(mat)

    icc_rows.append({
        'metric': metric,
        'n_proposals': mat.shape[0],
        'n_raters': mat.shape[1],
        'icc_2_1': icc21,
        'icc_2_k': icc2k,
    })

icc_df = pd.DataFrame(icc_rows)
icc_df


### Interpretation: ICC Reliability

- ICC(2,1): reliability of a single rater score.
- ICC(2,k): reliability of averaged panel score.
- Rule-of-thumb: <0.5 poor, 0.5-0.75 moderate, 0.75-0.9 good, >0.9 excellent.


In [ ]:
# Purpose:
# Optional Krippendorff alpha (ordinal) reliability estimate with flexible missingness handling.

try:
    import krippendorff

    hr = human_shared[['proposal_id', 'reviewer_id', 'scientific_merit_and_innovation_score', 'feasibility_score', 'data_sources_and_limitations_score', 'open_science_compliance_score']].copy()
    hr['overall_shared'] = hr[['scientific_merit_and_innovation_score', 'feasibility_score', 'data_sources_and_limitations_score', 'open_science_compliance_score']].mean(axis=1)
    hr['rater'] = hr['reviewer_id'].astype(str).map(lambda x: f'human-reviewer-{x}')

    ar = ai_panel[['proposal_id', 'evaluator', 'overall_shared']].copy().rename(columns={'evaluator': 'rater'})

    all_ratings = pd.concat([
        hr[['proposal_id', 'rater', 'overall_shared']],
        ar[['proposal_id', 'rater', 'overall_shared']],
    ], ignore_index=True)

    rating_matrix = all_ratings.pivot_table(index='rater', columns='proposal_id', values='overall_shared', aggfunc='mean')

    alpha_ord = krippendorff.alpha(
        reliability_data=rating_matrix.to_numpy(dtype=float),
        level_of_measurement='ordinal'
    )

    print({'krippendorff_alpha_ordinal_overall_shared': alpha_ord, 'n_raters': rating_matrix.shape[0], 'n_proposals': rating_matrix.shape[1]})
except Exception as e:
    print('Krippendorff alpha skipped:', e)


### Interpretation: Krippendorff Alpha (Optional)

- Higher alpha indicates stronger agreement across mixed human+AI raters.
- If alpha is low despite moderate ICC, inspect whether disagreements are ordinally structured or model-specific.


## 17) Calibration vs human panel (bias slope/intercept + Bland-Altman)

**Purpose**
Assess calibration of AI ratings against human panel reference values.

**How this is done**
- Fit linear calibration models (`AI ~ Human`) per evaluator and metric.
- Report slope/intercept and explained variance.
- Visualize agreement structure via Bland-Altman plots.

**Why this method is appropriate**
Calibration diagnostics distinguish systematic over/under-scoring from random noise and reveal range-dependent bias.


In [ ]:
# Purpose:
# Regression-based calibration diagnostics (slope/intercept) versus human panel scores.

cal_rows = []

for evaluator in sorted(ai_panel['evaluator'].unique()):
    sub_ai = ai_panel[ai_panel['evaluator'] == evaluator].copy()
    merged = sub_ai.merge(human_panel, on='proposal_id', suffixes=('_ai', '_human'))

    for metric in SHARED_METRICS_4CAT + ['overall_shared']:
        x = merged[f'{metric}_human'].values
        y = merged[f'{metric}_ai'].values
        lr = linregress(x, y)
        cal_rows.append({
            'evaluator': evaluator,
            'metric': metric,
            'n_proposals': len(merged),
            'slope': lr.slope,
            'intercept': lr.intercept,
            'r_value': lr.rvalue,
            'r_squared': lr.rvalue ** 2,
            'p_value': lr.pvalue,
            'std_err': lr.stderr,
        })

calibration_df = pd.DataFrame(cal_rows)
calibration_df = add_bh_fdr(calibration_df, p_col='p_value', group_cols=['metric'])
calibration_df


### Interpretation: Calibration Regression

- Ideal calibration is **slope ≈ 1** and **intercept ≈ 0**.
- Slope < 1 suggests score compression; slope > 1 suggests exaggeration.
- Intercept offsets indicate systematic over/under-scoring relative to human panel.


In [ ]:
# Purpose:
# Bland-Altman agreement visualization for overall shared score by evaluator.

fig, axes = plt.subplots(1, 3, figsize=(21, 6), sharey=True)
for ax, evaluator in zip(axes, sorted(ai_panel['evaluator'].unique())):
    sub_ai = ai_panel[ai_panel['evaluator'] == evaluator].copy()
    merged = sub_ai.merge(human_panel, on='proposal_id', suffixes=('_ai', '_human'))

    mean_vals = (merged['overall_shared_ai'] + merged['overall_shared_human']) / 2
    diff_vals = merged['overall_shared_ai'] - merged['overall_shared_human']
    mdiff = diff_vals.mean()
    sd = diff_vals.std(ddof=1)

    ax.scatter(mean_vals, diff_vals, alpha=0.8)
    ax.axhline(mdiff, color='red', linestyle='--', label='Mean diff')
    ax.axhline(mdiff + 1.96 * sd, color='gray', linestyle=':')
    ax.axhline(mdiff - 1.96 * sd, color='gray', linestyle=':')
    ax.set_title(f'Bland-Altman: {evaluator}')
    ax.set_xlabel('Mean of AI and Human (overall_shared)')

axes[0].set_ylabel('AI - Human')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'quality_proxy_bland_altman_overall_shared.png', dpi=200, bbox_inches='tight')
plt.show()


### Interpretation: Bland-Altman Plots

- Mean line near 0 indicates low average bias.
- Narrow limits of agreement indicate better consistency.
- Trend with x-axis (mean score) indicates heteroscedastic/range-dependent bias.


## 18) Leniency/stringency with proposal fixed effects (all proposals)

**Purpose**
Estimate evaluator-specific bias while controlling for proposal-level baseline difficulty/quality.

**How this is done**
- OLS with proposal fixed effects for overall score.
- OLS with evaluator-by-metric interactions for criterion-specific bias patterns.

**Why this method is appropriate**
Proposal fixed effects absorb latent proposal quality, so evaluator coefficients better capture scoring tendency rather than proposal mix differences.


In [ ]:
# Purpose:
# Estimate evaluator leniency/stringency with proposal fixed effects to control
# for proposal-level heterogeneity.

try:
    import statsmodels.formula.api as smf

    fe_base = ai_df[['proposal_uid', 'evaluator', 'overall_score']].dropna().copy()
    model_leniency = smf.ols('overall_score ~ C(evaluator) + C(proposal_uid)', data=fe_base).fit(cov_type='HC3')

    print(model_leniency.summary().tables[1])

    fe_long = ai_df[['proposal_uid', 'evaluator'] + QUALITY_METRICS].melt(
        id_vars=['proposal_uid', 'evaluator'],
        value_vars=QUALITY_METRICS,
        var_name='metric',
        value_name='score'
    ).dropna(subset=['score'])

    model_leniency_metric = smf.ols('score ~ C(evaluator) * C(metric) + C(proposal_uid)', data=fe_long).fit(cov_type='HC3')
    print(model_leniency_metric.summary().tables[1])
except Exception as e:
    print('statsmodels FE models skipped:', e)


### Interpretation: Leniency/Stringency Fixed-Effects Models

- Evaluator coefficients reflect leniency/stringency after controlling proposal fixed effects.
- Interaction terms with `metric` reveal criterion-specific strictness.
- Use this section to support adjusted comparisons when evaluator bias is non-negligible.


## 19) Export tables

This section writes all analysis artifacts (mapping diagnostics, test tables, robustness outputs, bias/proxy/reliability results) to `results/figures/quality/` for downstream reporting and manuscript integration.


In [ ]:
# Purpose:
# Persist all analysis outputs for reporting, reproducibility, and manuscript figures/tables.

mapping_df.to_csv(OUTPUT_DIR / 'quality_matching_map_exact_fuzzy.csv', index=False)
pair_df.to_csv(OUTPUT_DIR / 'quality_similarity_pairs.csv', index=False)
matching_sensitivity_df.to_csv(OUTPUT_DIR / 'quality_similarity_matching_sensitivity.csv', index=False)

sim_stats_overall_df.to_csv(OUTPUT_DIR / 'quality_similarity_mw_cliffs_overall.csv', index=False)
sim_stats_hai_by_model_df.to_csv(OUTPUT_DIR / 'quality_similarity_mw_cliffs_human_ai_by_model.csv', index=False)
sim_stats_aiai_by_pair_df.to_csv(OUTPUT_DIR / 'quality_similarity_mw_cliffs_ai_ai_by_model_pair.csv', index=False)

summary_overall_df.to_csv(OUTPUT_DIR / 'quality_summary_overall_by_author_group.csv', index=False)
pairwise_df.to_csv(OUTPUT_DIR / 'quality_pairwise_mw_cliffs_all_metrics_proposal_level.csv', index=False)
robust_quality_df.to_csv(OUTPUT_DIR / 'quality_robust_bootstrap_permutation_key_comparisons.csv', index=False)

eval_stats.to_csv(OUTPUT_DIR / 'quality_evaluator_overall_stats_clean.csv')
self_pref_df.to_csv(OUTPUT_DIR / 'quality_self_preference_tests_overall.csv', index=False)
self_pref_metric_df.to_csv(OUTPUT_DIR / 'quality_self_preference_tests_by_metric.csv', index=False)

proxy_validity_df.to_csv(OUTPUT_DIR / 'quality_proxy_validity_metrics.csv', index=False)
rank_agreement_df.to_csv(OUTPUT_DIR / 'quality_proxy_rank_agreement.csv', index=False)
icc_df.to_csv(OUTPUT_DIR / 'quality_proxy_icc.csv', index=False)
calibration_df.to_csv(OUTPUT_DIR / 'quality_proxy_calibration_regression.csv', index=False)

print('Saved outputs to:', OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob('quality_*')):
    print('-', p.name)


## Notes

- Run this notebook top-to-bottom; dependencies are linear by design.
- Interpretation cells below each result explain what to look for and how to report it.
- For significance decisions, prioritize FDR-adjusted `q_value` over raw `p_value`.
- For practical relevance, report effect size magnitude and direction alongside significance.
